# Rosati α-β — Chain separability (sweep workflow)

Replicates the **final CD4/CD8 sweep workflow**, but with the label = **chain (alpha vs beta)**
instead of CD4/CD8. Same canonical metric (`ref_size_sweep_auroc`), same set of plots.

**Note:** separating alpha from beta is expected to be near-trivial (~1.0) — different loci, different
CDR3 lengths. This is the **sanity-check baseline** confirming the pipeline works. The interesting
labels (donor, group) come later as follow-up questions.

**Parameterized for both UMI versions:** set `UMI = 1` or `UMI = 2` in cell 1 and re-run.
Data: Rosati Study 1, bulk paired blood, embedded clouds (per (donor, chain)).

## 1 — UMI parameter + paths

In [ ]:
# Run with UMI=1, then change to UMI=2 and re-run. Each version uses its own embedded folder.
UMI = 1   # <--- change to 2 for the other version

import sys, re, glob, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPERTOIRE_DIR = '/home/immunologylab/bioinformatics/analysis/tcr_repertoire/scripts/repertoire'
CLOUDS_DIR     = f'/home/immunologylab/bioinformatics/analysis/data/processed/rosati/clouds_embedded_umi{UMI}'
LANDMARKS      = '/home/immunologylab/bioinformatics/analysis/data/processed/descriptors/landmarks_beta.npz'
sys.path.insert(0, REPERTOIRE_DIR)
import rep_data, rep_descriptors as rd, rep_metrics as rm

print(f'UMI = {UMI}  ->  {CLOUDS_DIR}')
print('embedded files:', len(glob.glob(f'{CLOUDS_DIR}/*.parquet')))

## 2 — Load clouds + compute mean+cov descriptor (one point per cloud)
Filenames are like `CD_100_A.par.parquet` → donor `CD_100`, chain `A` (alpha) / `B` (beta).

In [ ]:
def parse_stem(fname):
    base = os.path.basename(fname).replace('.par.parquet','').replace('.parquet','')
    donor = '_'.join(base.split('_')[:-1])   # 'CD_100'
    chain = base.split('_')[-1]              # 'A' or 'B'
    chain_label = 'alpha' if chain == 'A' else 'beta'
    group = donor.split('_')[0]              # CD / Healthy / UC
    return donor, chain_label, group

files = sorted(glob.glob(f'{CLOUDS_DIR}/*.parquet'))
EMB = [f'e{i}' for i in range(128)]

V, chain_lab, donor_lab, group_lab = [], [], [], []
for f in files:
    df = pd.read_parquet(f, columns=EMB + ['w_log'])
    Z = df[EMB].to_numpy(np.float32)
    Z /= (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-8)
    w = df['w_log'].to_numpy(np.float32)
    V.append(rd.mean_cov_weighted_np(Z, w))
    donor, chain, group = parse_stem(f)
    chain_lab.append(chain); donor_lab.append(donor); group_lab.append(group)

V = np.vstack(V)
chain_lab = np.array(chain_lab); donor_lab = np.array(donor_lab); group_lab = np.array(group_lab)

print('clouds:', V.shape[0], '| descriptor dim:', V.shape[1])
print('chains:', pd.Series(chain_lab).value_counts().to_dict())
print('groups:', pd.Series(group_lab).value_counts().to_dict())
print('donors with both chains:', (pd.Series(donor_lab).value_counts()==2).sum())

## 3 — Main sweep: label = chain (alpha vs beta)
Do the nearest neighbours of a cloud share its chain? (sanity check; expect high).

In [ ]:
ref_sizes = [1, 2, 5, 10, 15, 20, 25]
sweep = rm.ref_size_sweep_auroc(V, chain_lab, ref_sizes, n_draws=100, seed=0)
ys = [sweep[r] for r in ref_sizes]

fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.plot(ref_sizes, ys, marker='o', markersize=9, linewidth=2.5, color='#065A82',
        markerfacecolor='#065A82', markeredgecolor='white', markeredgewidth=1.5, label='mean+cov')
for x, y in zip(ref_sizes, ys):
    ax.annotate(f'{y:.3f}', (x, y), textcoords='offset points', xytext=(0, 12),
                ha='center', fontsize=9, color='#21295C', fontweight='bold')
ax.axhline(0.5, color='gray', ls='--', lw=1, alpha=0.6)
ax.text(ref_sizes[-1], 0.515, 'chance (0.5)', ha='right', fontsize=9, color='gray')
ax.set_xlabel('Reference-set size  (number of reference clouds)', fontsize=12, fontweight='bold')
ax.set_ylabel('Mean ROC-AUC  (alpha vs beta)', fontsize=12, fontweight='bold')
ax.set_title(f'Chain separability (alpha vs beta) vs reference-set size  [UMI≥{UMI}]', fontsize=12)
ax.set_xticks(ref_sizes); ax.set_ylim(0.45, 1.02); ax.grid(True, alpha=0.25)
ax.legend(loc='lower right', fontsize=10)
plt.tight_layout(); plt.show()

print('ref_size : mean ROC-AUC')
for r in ref_sizes:
    print(f'   {r:>2}    :  {sweep[r]:.4f}')

## 4 — Sweep by descriptor (mean+cov vs occupancy)

In [ ]:
curves = {'mean+cov': ys}
Vo = None
if os.path.exists(LANDMARKS):
    protos = np.load(LANDMARKS)['centroids']
    Vo = []
    for f in files:
        df = pd.read_parquet(f, columns=EMB + ['w_log'])
        Z = df[EMB].to_numpy(np.float32); Z /= (np.linalg.norm(Z,axis=1,keepdims=True)+1e-8)
        w = df['w_log'].to_numpy(np.float32)
        Vo.append(rd.weighted_occupancy(Z, w, protos, tau=0.1))
    Vo = np.vstack(Vo)
    sweep_o = rm.ref_size_sweep_auroc(Vo, chain_lab, ref_sizes, n_draws=100, seed=0)
    curves['occupancy'] = [sweep_o[r] for r in ref_sizes]

    fig, ax = plt.subplots(figsize=(8.5, 5.5))
    colors = {'mean+cov':'#065A82', 'occupancy':'#C0392B'}
    for name, yv in curves.items():
        ax.plot(ref_sizes, yv, marker='o', markersize=8, linewidth=2.3, label=name, color=colors[name])
    ax.axhline(0.5, color='gray', ls='--', lw=1, alpha=0.6)
    ax.set_xlabel('Reference-set size  (number of reference clouds)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Mean ROC-AUC  (alpha vs beta)', fontsize=12, fontweight='bold')
    ax.set_title(f'Chain sweep by descriptor  [UMI≥{UMI}]', fontsize=12)
    ax.set_xticks(ref_sizes); ax.set_ylim(0.45, 1.02); ax.grid(True, alpha=0.25)
    ax.legend(loc='lower right', fontsize=10)
    plt.tight_layout(); plt.show()
    print('occupancy:', {r: round(sweep_o[r],4) for r in ref_sizes})
else:
    print('landmarks not found - mean+cov only')

## 5 — Full-depth vs rarefied (depth-confound control)
Subsample every cloud to a common K and check the sweep is unchanged.

In [ ]:
# common depth = min clonotypes across clouds (bounded so it isn't dragged to a tiny value)
sizes = [len(pd.read_parquet(f, columns=['w_log'])) for f in files]
K = int(np.percentile(sizes, 5))   # 5th percentile as a robust floor
print('rarefying to K =', K, '| min cloud =', min(sizes))

def descriptor_rarefied(f, K, seed):
    df = pd.read_parquet(f, columns=EMB + ['w_log'])
    if len(df) > K:
        df = df.sample(K, random_state=seed)
    Z = df[EMB].to_numpy(np.float32); Z /= (np.linalg.norm(Z,axis=1,keepdims=True)+1e-8)
    w = df['w_log'].to_numpy(np.float32); w = w / w.sum()
    return rd.mean_cov_weighted_np(Z, w)

n_seeds = 5
rare = {r: [] for r in ref_sizes}
for seed in range(n_seeds):
    Vr = np.vstack([descriptor_rarefied(f, K, seed) for f in files])
    sw = rm.ref_size_sweep_auroc(Vr, chain_lab, ref_sizes, n_draws=100, seed=seed)
    for r in ref_sizes: rare[r].append(sw[r])
rare_mean = [np.mean(rare[r]) for r in ref_sizes]
rare_std  = [np.std(rare[r])  for r in ref_sizes]

fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.plot(ref_sizes, ys, marker='o', markersize=8, linewidth=2.5, color='#065A82', label='full depth')
ax.errorbar(ref_sizes, rare_mean, yerr=rare_std, marker='s', markersize=7, linewidth=2.5,
            color='#C0392B', label=f'rarefied to K={K}', capsize=3)
for x, y in zip(ref_sizes, ys):
    ax.annotate(f'{y:.3f}', (x, y), textcoords='offset points', xytext=(0, 12), ha='center', fontsize=8.5, color='#065A82', fontweight='bold')
for x, y in zip(ref_sizes, rare_mean):
    ax.annotate(f'{y:.3f}', (x, y), textcoords='offset points', xytext=(0, -16), ha='center', fontsize=8.5, color='#C0392B', fontweight='bold')
ax.axhline(0.5, color='gray', ls='--', lw=1, alpha=0.6)
ax.set_xlabel('Reference-set size  (number of reference clouds)', fontsize=12, fontweight='bold')
ax.set_ylabel('Mean ROC-AUC  (alpha vs beta)', fontsize=12, fontweight='bold')
ax.set_title(f'Chain sweep: full depth vs rarefied  [UMI≥{UMI}]', fontsize=12)
ax.set_xticks(ref_sizes); ax.set_ylim(0.45, 1.02); ax.grid(True, alpha=0.25)
ax.legend(loc='lower right', fontsize=10)
plt.tight_layout(); plt.show()

## 6 — Sweep vs depth-only baseline
Line = AUROC of chain from cloud size alone (no embedding).

In [ ]:
from sklearn.metrics import roc_auc_score
n_clono = np.array(sizes)
y_beta = (chain_lab == 'beta').astype(int)
# depth-only: does cloud size predict chain? (beta tends deeper)
depth_auc = roc_auc_score(y_beta, n_clono)
depth_auc = max(depth_auc, 1 - depth_auc)   # direction-agnostic
print(f'depth-only baseline: {depth_auc:.4f}')

fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.plot(ref_sizes, ys, marker='o', markersize=9, linewidth=2.5, color='#065A82', label='mean+cov (foundation)')
for x, y in zip(ref_sizes, ys):
    ax.annotate(f'{y:.3f}', (x, y), textcoords='offset points', xytext=(0, 12), ha='center', fontsize=9, color='#21295C', fontweight='bold')
ax.axhline(depth_auc, color='#C0392B', ls='--', lw=1.8, alpha=0.8, label=f'depth-only baseline ({depth_auc:.3f})')
ax.axhline(0.5, color='gray', ls=':', lw=1, alpha=0.6)
ax.set_xlabel('Reference-set size  (number of reference clouds)', fontsize=12, fontweight='bold')
ax.set_ylabel('Mean ROC-AUC  (alpha vs beta)', fontsize=12, fontweight='bold')
ax.set_title(f'Chain sweep vs depth-only baseline  [UMI≥{UMI}]', fontsize=12)
ax.set_xticks(ref_sizes); ax.set_ylim(0.45, 1.02); ax.grid(True, alpha=0.25)
ax.legend(loc='lower right', fontsize=10)
plt.tight_layout(); plt.show()

## 7 — Sweep vs V-gene baseline
Three curves: mean+cov, occupancy, V-gene usage.

In [ ]:
vgene_data = []
for f in files:
    df = pd.read_parquet(f, columns=['v_gene','w_log'])
    vgene_data.append((df['v_gene'].to_numpy(), df['w_log'].to_numpy()))
vocab = rm.build_vgene_vocab([vg for vg,_ in vgene_data])
V_vg = np.vstack([rm.vusage_vector(vg, w, vocab) for vg,w in vgene_data])
sweep_vg = rm.ref_size_sweep_auroc(V_vg, chain_lab, ref_sizes, n_draws=100, seed=0)
ys_vg = [sweep_vg[r] for r in ref_sizes]

fig, ax = plt.subplots(figsize=(9, 5.8))
ax.plot(ref_sizes, ys, marker='o', markersize=8, linewidth=2.5, color='#065A82', label='mean+cov (foundation)')
if Vo is not None:
    ax.plot(ref_sizes, curves['occupancy'], marker='^', markersize=8, linewidth=2.3, color='#1C7293', label='occupancy')
ax.plot(ref_sizes, ys_vg, marker='s', markersize=8, linewidth=2.3, color='#C0392B', label='V-gene usage (baseline)')
ax.axhline(0.5, color='gray', ls='--', lw=1, alpha=0.6)
ax.set_xlabel('Reference-set size  (number of reference clouds)', fontsize=12, fontweight='bold')
ax.set_ylabel('Mean ROC-AUC  (alpha vs beta)', fontsize=12, fontweight='bold')
ax.set_title(f'Chain sweep: foundation descriptors vs V-gene baseline  [UMI≥{UMI}]', fontsize=12)
ax.set_xticks(ref_sizes); ax.set_ylim(0.45, 1.02); ax.grid(True, alpha=0.25)
ax.legend(loc='lower right', fontsize=10)
plt.tight_layout(); plt.show()
print('V-gene:', {r: round(sweep_vg[r],4) for r in ref_sizes})

## UMI≥2 version (re-run of all sweeps with the stricter threshold)